In [1]:
#libraries
# from sqlalchemy import text, create_engine
# from sklearn.preprocessing import RobustScaler, MinMaxScaler
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from momentfm import MOMENTPipeline
from tqdm import tqdm

c:\Python Projects\Late Dispatches\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Python Projects\Late Dispatches\venv\Lib\site-packages\transformers\utils\generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


In [ ]:
#hyperparameters
# obs_start_date = pd.Timestamp(2025, 1, 1).strftime('%Y-%m-%d') # first observation date
# obs_end_date = pd.Timestamp(2025, 12, 20).strftime('%Y-%m-%d') # last observation date


rufus_engine = create_engine(r"sqlite:///C:/Python Projects/local.db")
with rufus_engine.connect() as conn:
    trans = pd.read_sql(text(f'''
WITH non_phantom_sord AS (
    SELECT s.sord_order, s.sord_qty_req, s.sord_qty_desp, s.sord_date_req, s.sord_datetime,
        MIN(COALESCE(strc.rufus_component_id, s.rufus_stkno_id)) AS rufus_stkno_id -- only 1 product_id ensure no rows are duplicated
    FROM sord s
    -- join phantom parts
    LEFT JOIN stck ON stck.rufus_stkno_id = s.rufus_stkno_id AND stck.stck_user1 != ' ' AND stck.stck_prod_group != 10003
    -- get phantom components (stock items)
    LEFT JOIN strc ON strc.rufus_product_id = stck.rufus_stkno_id
    -- get stck info about components
    LEFT JOIN stck phantom_comp ON phantom_comp.rufus_stkno_id = strc.rufus_component_id
    WHERE (phantom_comp.stck_prod_group NOT LIKE '9%' -- filter phantom parts incorrectly set up against black products
        OR phantom_comp.stck_prod_group IS NULL)
    GROUP BY s.sord_order, s.sord_qty_req, s.sord_qty_desp, s.sord_date_req, s.sord_datetime
),
acaud_sord AS (
SELECT
    DATETIME(acaud_sys_date, '+' || (acaud_post_time / 10000) || ' hours') AS datetime_post_hour,
    DATETIME(post_datetime) AS datetime_post,
    DATETIME(sord_datetime) AS datetime_sord,
    DATE(sord_date_req) AS day_req,
    a.rufus_stkno_id,
    acaud_qty AS qty,
    acaud_option AS portal,
    acaud_open_balance + acaud_qty AS on_hand,
    acaud_job,
    CASE WHEN stck_prod_group IN (99, 99999) THEN 1 ELSE 0 END AS wip,
    CASE WHEN stck_prod_group = 10009 THEN 1 ELSE 0 END AS st,
    sord_order, -- composite primary key with rufus_stkno_id
    sord_qty_req, -- duplicated with the join to acuad!
    sord_qty_desp -- used in deployment to see open orders               
FROM acaud a
LEFT JOIN non_phantom_sord s ON s.rufus_stkno_id = a.rufus_stkno_id AND a.acaud_ref1 = s.sord_order
JOIN stck ON stck.rufus_stkno_id = a.rufus_stkno_id
-- LEFT JOIN lcus ON lcus_code = sord_lcus WHERE lcus_delivery_flag NOT IN ('B', 'R')
WHERE
    stck_prod_group = 10001 -- can swap for : prod_groups when not using sqlite
    AND (sord_order NOT LIKE 'C%' OR sord_order IS NULL)
    AND (sord_date_req BETWEEN '{obs_start_date}' AND '{obs_end_date}' OR sord_date_req IS NULL)
    AND (sord_datetime BETWEEN '{obs_start_date}' AND '{obs_end_date}' OR sord_datetime IS NULL)
    AND post_datetime BETWEEN '{obs_start_date}' AND '{obs_end_date}'
    ),

orders AS (
SELECT
    datetime_sord AS datetime_post, -- transaction time
    day_req, -- when order was due - only relevant to orders and despatch
    MAX(datetime_post) AS datetime_order_desp, -- when order was dispatched - only relevant to orders
    1 AS loc, -- process location
    rufus_stkno_id,
    1 AS num_transactions,
    sord_qty_req AS qty,
    NULL AS portal,
    NULL AS on_hand,
    NULL as wip_on_hand
FROM acaud_sord
WHERE day_req IS NOT NULL
GROUP BY sord_order, rufus_stkno_id, datetime_sord, sord_qty_req, day_req  -- grouped as can have multiple acaud transactions per sord order
    ),
                             
wip_trans AS (
SELECT
    datetime_post,
    NULL AS day_req,
    NULL AS datetime_order_desp,
    2 AS loc,
    strc.rufus_product_id AS rufus_stkno_id,
    1 AS num_transactions,
    qty,
    portal,
    NULL AS on_hand,
    on_hand AS wip_on_hand
FROM acaud_sord
JOIN strc ON strc.rufus_component_id = acaud_sord.rufus_stkno_id
WHERE wip = 1
),

st_prod AS ( --grouped as barcodes scanned in one by one
SELECT
    datetime_post_hour AS datetime_post,
    day_req,
    NULL AS datetime_order_desp,
    3 AS loc,
    rufus_stkno_id,
    COUNT(*) AS num_transactions,
    sum(qty) AS qty,
    portal,
    MAX(on_hand) AS on_hand,
    NULL as wip_on_hand
FROM acaud_sord
WHERE st = 1 AND qty > 0
GROUP BY rufus_stkno_id, datetime_post_hour, portal, day_req
    ),

finished_prod AS (
SELECT
    datetime_post,
    day_req,
    NULL AS datetime_order_desp,
    3 AS loc,
    rufus_stkno_id,
    1 AS num_transactions,
    qty,
    portal,
    on_hand,
    NULL AS wip_on_hand
FROM acaud_sord
WHERE wip = 0 AND st = 0 AND qty > 0
),

despatches AS ( -- grouped as multiple orders can be posted at once
SELECT
    MAX(datetime_post) AS datetime_post,
    day_req,
    NULL AS datetime_order_desp,
    4 AS loc,
    rufus_stkno_id,
    COUNT(*) AS num_transactions,
    sum(qty) AS qty,
    portal,
    MIN(on_hand) AS on_hand,
    NULL AS wip_on_hand
FROM acaud_sord
WHERE wip = 0 AND qty < 0
GROUP BY acaud_job, rufus_stkno_id, portal, datetime_post_hour, day_req
),

union_trans AS (
SELECT * FROM orders
UNION ALL SELECT * FROM wip_trans
UNION ALL SELECT * FROM st_prod
UNION ALL SELECT * FROM finished_prod
UNION ALL SELECT * FROM despatches
)

SELECT
    -- categorical variables first for encoding
    loc,
    portal,
    datetime_post,
    day_req,
    datetime_order_desp,
    rufus_stkno_id,
    num_transactions,
    qty,
    on_hand,
    wip_on_hand
FROM union_trans
ORDER BY rufus_stkno_id, datetime_post
'''), con = conn, index_col='rufus_stkno_id')



#trans preprocessing

#encode portal as integers for embedding
portal_codings = {v:i + 1 for i, v in enumerate(trans['portal'].unique())}
trans['portal'] = trans['portal'].map(portal_codings)

#forward fill stock columns
forward_fill_cols = ['on_hand', 'wip_on_hand']
for col in forward_fill_cols:
    trans[col] = trans.groupby(level=0)[col].ffill().fillna(0)

#datetime conversions for transformer

trans['datetime_post'] = pd.to_datetime(trans['datetime_post'])

#this is y unsure how to use this yet
# would be better if there is any late dispatch in next two days
trans['late'] = (pd.to_datetime(trans['datetime_order_desp']).dt.normalize() > pd.to_datetime(trans['datetime_post'])).astype(int)

#convert day_req to days between order and required date
trans['day_req'] = (pd.to_datetime(trans['day_req']) - trans['datetime_post'].dt.normalize()).dt.days


trans['datetime_post'] = (
    trans['datetime_post'] - trans['datetime_post'].groupby(level=0).shift(1)
).dt.total_seconds() / 86400

trans.drop(columns = ['datetime_order_desp'], inplace=True)

from torch.utils.data import DataLoader, Dataset

trans.sort_index(inplace=True)

temporal_array = trans.fillna(0).to_numpy() #convert to numpy
#fillna(0) ok here? Only due date should be na for internal transactions so probs all good

#scale data
temporal_scaler = RobustScaler(with_centering=False) # 0 is important so no centering
temporal_scaler.fit(temporal_array[:, 2:]) # in future should be just on training data
temporal_array[:, 2:] = temporal_scaler.transform(temporal_array[:, 2:]) #first two cols categorical integers for embedding

# index of trans dataframe see where stkno groupings are
sequences = [group.to_numpy() for _, group in trans.groupby(level=0)]

#shape of input array
num_sequences = len(sequences)
num_features = temporal_array.shape[1]

# allocate zero arrays for masks
padded = np.zeros((num_sequences, context_window, num_features), dtype=np.float32)
mask = np.zeros((num_sequences, context_window), dtype=bool)

for i, seq in enumerate(sequences):
    L = len(seq)

    if L > context_window:
        seq = seq[-context_window:]  # truncate to most recent
        L = context_window

    padded[i, -L:] = seq
    mask[i, -L:] = True

X = torch.tensor(padded[..., :-1 ,:-1].permute(0, 2, 1), dtype=torch.float32) #all but last column
Y = torch.tensor(padded[..., 1: ,-1], dtype=torch.float32) # last column offset by 1
mask = torch.tensor(mask[..., :-1], dtype=torch.bool)


train_data = Dataset(X, Y, mask)

In [3]:
class FoundationDataset(Dataset):
    def __init__(self, X, dense, mask, Y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.dense = torch.tensor(dense, dtype=torch.float32)
        self.mask = torch.tensor(mask, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)

    def __len__(self):
        return len(self.Y)

    def __getitem__(self, idx):
        return {
            "X": self.X[idx],
            "dense": self.dense[idx],
            "mask": self.mask[idx],
            "Y": self.Y[idx]
        }

In [4]:
data = np.load(r"C:\Python Projects\ML Transactions\data_cache\obs_2025.npz")

temporal_test = data["temporal_test"]

train_dataset = FoundationDataset(temporal_test, data["dense_test"], data["mask_test"], data["Y_test"]) #change test to train if run preprocessing again

In [5]:
import torch.nn as nn

class embModel(nn.Module):
    def __init__(self, foundation_model, loc_dims, portal_dims):
        super().__init__()
        self.foundation_model = foundation_model

        for name, param in self.foundation_model.named_parameters():
            if name in ['head.linear.weight', 'head.linear.bias']:
                param.requires_grad = True
            else:
                param.requires_grad = False
        
        #columns to embed from left of X input
        self.emb_layers = nn.ModuleList([
            nn.Embedding(loc_dims, 1),
            nn.Embedding(portal_dims, 1)
        ])

    def forward(self, x, mask):
        # x: [B, T, C]
        embs = []
        
        # embed first channels
        for i, layer in enumerate(self.emb_layers):
            emb = layer(x[:, :, i].round().long())   # [B, T, emb_dim]
            embs.append(emb)

        # concatenate embeddings and remaining channels
        x = torch.cat(embs + [x[:, :, len(self.emb_layers):]], dim=2)
        
        # swap final columns for foundation model
        x = x.permute(0, 2, 1)

        # pass through foundation model
        out = self.foundation_model(x_enc=x, attention_mask=mask)
        
        return out
               

In [14]:

foundation_model = MOMENTPipeline.from_pretrained(
    "AutonLab/MOMENT-1-large", 
    model_kwargs={
        'task_name': 'classification',
        'n_channels': temporal_test.shape[-1], #if want to embed with more than 1 channel this must increase here
        'num_class': 1 #binary classifier for now
    },
)
foundation_model.init()

loc_dims = 6
portal_dims = 11
batch_size = 256 #transaction histories to run in parallel

model = embModel(foundation_model, loc_dims, portal_dims)

criterion = torch.nn.BCEWithLogitsLoss() # loss fn
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

#lower triangle causal mask - used if training multiple classifiers on same history
# causal_mask = torch.tril(torch.ones((context_window, context_window), dtype=torch.bool)).unsqueeze(0).expand(batch_size, -1, -1)

# training data shape: [batches, preditors, context length (512)]
# Y shape : [batches, ]

device = torch.device("xpu")
model = model.to(device)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=0) 

for batch in tqdm(train_dataloader, desc="Training"):
    #use gpu for faster computation
    X= batch['X'].to(device, non_blocking = True)
    mask= batch['mask'].to(device, non_blocking = True)
    Y  = batch['Y'].to(device, non_blocking = True)

    # forward [batch_size, n_channels, context length]
    output = model(X, mask) # get logits from custom forward pass
    
    # backward
    loss = criterion(output.logits, Y) # compare logits to Y
    optimizer.zero_grad() # reset gradients
    loss.backward() # calculate gradients with backwards pass
    optimizer.step() # update weights
    
    print(f"loss: {loss.item():.3f}")

Training:   0%|          | 0/1924 [00:00<?, ?it/s]c:\Python Projects\Late Dispatches\venv\Lib\site-packages\torch\_dynamo\eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


torch.Size([256, 30, 19])


Training:   0%|          | 1/1924 [00:41<21:59:03, 41.16s/it]

loss: 0.687
torch.Size([256, 30, 19])


Training:   0%|          | 2/1924 [01:25<23:07:43, 43.32s/it]

loss: 0.683
torch.Size([256, 30, 19])


Training:   0%|          | 2/1924 [02:12<35:27:07, 66.40s/it]


KeyboardInterrupt: 